# Linearization residual explorer

This notebook loads the cached residual summary, converts it to a tidy DataFrame, and creates publication-style plots for layer-wise residual dynamics.

The goal is to make the figure easy to tune: choose the color palette, error band, axis limits, and highlight the most non-linear layers.

In [ ]:
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Keep the notebook robust regardless of where it is launched from.
candidates = [
    Path.cwd(),
    Path.cwd() / "LinearLLMDynamic" / "parking" / "linearization_error_explore",
    Path.cwd().parent / "LinearLLMDynamic" / "parking" / "linearization_error_explore",
]
NOTEBOOK_DIR = next((p for p in candidates if (p / "cache").exists()), Path.cwd())
DATA_PATH = NOTEBOOK_DIR / "cache" / "qwen_residual_summary.json"
PLOTS_DIR = NOTEBOOK_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True, parents=True)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Summary path: {DATA_PATH}")

In [ ]:
with DATA_PATH.open("r", encoding="utf-8") as f:
    summary = json.load(f)

df = pd.DataFrame(summary).sort_values("layer").reset_index(drop=True)
df.head()

## Figure 1: Mean residual vs layer

This is the clearest view for seeing where the local linear approximation becomes worse. The shaded band shows one standard deviation across the sample set.

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 180,
    "savefig.dpi": 220,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "font.size": 11,
})

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(
    df["layer"],
    df["mean_relative_residual"],
    color="#0f4c81",
    linewidth=2.8,
    marker="o",
    markersize=6,
    label="Mean relative residual",
)
ax.fill_between(
    df["layer"],
    df["mean_relative_residual"] - df["std_relative_residual"],
    df["mean_relative_residual"] + df["std_relative_residual"],
    color="#0f4c81",
    alpha=0.18,
    label="±1 std",
)
ax.axhline(0.5, color="#4d4d4d", linestyle="--", linewidth=1, alpha=0.8)
ax.set_title("Linearization residual across transformer layers")
ax.set_xlabel("Layer index")
ax.set_ylabel("Mean relative residual")
ax.grid(alpha=0.25)
ax.legend(frameon=False)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "residual_mean_curve.png", bbox_inches="tight")
plt.show()

## Figure 2: Highlight the worst layers

Sort the layers by mean residual and plot the top-$k$ offenders. This is useful for diagnosing where the linear approximation breaks most sharply.

In [ ]:
top_k = 8
ranked = df.sort_values("mean_relative_residual", ascending=False).head(top_k).copy()

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("viridis", n_colors=len(ranked))
ax.barh(ranked["layer"].astype(str), ranked["mean_relative_residual"], color=colors, edgecolor="black", linewidth=0.5)
ax.invert_yaxis()
ax.set_title(f"Top {top_k} layers by mean relative residual")
ax.set_xlabel("Mean relative residual")
ax.set_ylabel("Layer index")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "residual_top_layers.png", bbox_inches="tight")
plt.show()

## Figure 3: Diagnostics table

A compact summary of the score spread helps when you want to compare layers numerically.

In [ ]:
display_cols = [
    "layer",
    "mean_relative_residual",
    "std_relative_residual",
    "min_relative_residual",
    "max_relative_residual",
]
summary_table = df[display_cols].copy()
summary_table["mean_relative_residual"] = summary_table["mean_relative_residual"].map(lambda x: f"{x:.3f}")
summary_table["std_relative_residual"] = summary_table["std_relative_residual"].map(lambda x: f"{x:.3f}")
summary_table["min_relative_residual"] = summary_table["min_relative_residual"].map(lambda x: f"{x:.3f}")
summary_table["max_relative_residual"] = summary_table["max_relative_residual"].map(lambda x: f"{x:.3f}")
summary_table

## Optional: style tuning

Copy and tweak the parameters below to explore a cleaner final figure for a paper or slide.

In [ ]:
# Example tuning knobs
fig_w, fig_h = 10, 5
line_color = "#1f77b4"
fill_color = "#1f77b4"
highlight_layers = [23, 1, 5, 4, 0]

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.plot(
    df["layer"],
    df["mean_relative_residual"],
    color=line_color,
    linewidth=2.5,
    marker="o",
    markersize=5,
)
for layer in highlight_layers:
    y = float(df.loc[df["layer"] == layer, "mean_relative_residual"].iloc[0])
    ax.scatter([layer], [y], color="red", s=50, zorder=5)
    ax.annotate(str(layer), (layer, y), textcoords="offset points", xytext=(4, 4), color="red")

ax.set_xlabel("Layer")
ax.set_ylabel("Relative residual")
ax.set_title("Tuned residual view")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Paper-ready residual figure + hidden-state trajectories

This notebook produces a cleaner, publication-style residual plot and a low-dimensional state-space view of hidden-state trajectories for several prompts. The goal is to highlight how representation drift evolves across layers for different semantic directions.


In [ ]:
# State-space view of hidden-state trajectories for multiple prompts.
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map='auto',
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Benchmark / dataset note:
# - The residual summary is computed from a small SST-2 style sentence subset.
# - The hidden-state trajectory figure uses a custom sentiment prompt set, intentionally created to visualize language directions.
# - This is not a separate benchmark dataset; it is a focused prompt set modeled after sentiment tasks.
benchmarks = [
    ('SST-2 / GLUE', 'https://huggingface.co/datasets/glue/viewer/sst2'),
]

prompt_specs = [
    ('Prompt set 1 (positive)', 'I absolutely loved this movie; the acting was brilliant and the story was moving.'),
    ('Prompt set 2 (positive)', 'The performance was outstanding, and the atmosphere felt warm and uplifting.'),
    ('Prompt set 3 (positive)', 'The service was surprisingly smooth and the staff were helpful.'),
    ('Prompt set 4 (positive)', 'This product is excellent, reliable, and genuinely improved my day.'),
    ('Prompt set 5 (positive)', 'The experience was delightful from start to finish and felt thoughtfully designed.'),
    ('Prompt set 6 (negative)', 'This was a terrible experience, and I hated every minute of it.'),
    ('Prompt set 7 (negative)', 'The product arrived broken and the support was completely useless.'),
    ('Prompt set 8 (negative)', 'I felt frustrated and disappointed by the entire process and quality.'),
    ('Prompt set 9 (negative)', 'Everything felt slow, confusing, and badly handled from beginning to end.'),
    ('Prompt set 10 (negative)', 'The service was awful and the final result left me deeply dissatisfied.'),
]

all_trajs = []
all_labels = []
all_prompts = []

for label, prompt in prompt_specs:
    encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=32)
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        outputs = model(**encoded, output_hidden_states=True, return_dict=True, use_cache=False)
    states = []
    for layer_idx in range(len(outputs.hidden_states) - 1):
        vec = outputs.hidden_states[layer_idx][0, -1, :].float().cpu().numpy()
        states.append(vec)
    traj = np.stack(states, axis=0)
    all_trajs.append(traj)
    all_labels.append(label)
    all_prompts.append(prompt)

# 3D PCA over the full trajectory stack.
X = np.vstack(all_trajs)
X2 = X - X.mean(axis=0, keepdims=True)
proj = PCA(n_components=3, random_state=0)
coords3 = proj.fit_transform(X2)
explained = proj.explained_variance_ratio_
print('PCA explained variance ratio for 3 PCs:', explained)
print('Cumulative explained variance for 3 PCs:', explained.sum())

# Build a 3D prompt-annotated trajectory figure.
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
start = 0
for traj, label, prompt in zip(all_trajs, all_labels, all_prompts):
    n = len(traj)
    c = coords3[start:start + n]
    pos = 'positive' in label.lower()
    color = '#1f77b4' if pos else '#d62728'
    ax.plot(c[:, 0], c[:, 1], c[:, 2], color=color, linewidth=2, marker='o', markersize=4, alpha=0.9)
    ax.scatter([c[0, 0]], [c[0, 1]], [c[0, 2]], color='black', s=18, zorder=5)
    ax.scatter([c[-1, 0]], [c[-1, 1]], [c[-1, 2]], color='black', s=24, zorder=5)
    snippet = (prompt[:36] + '...') if len(prompt) > 36 else prompt
    ax.text(c[-1, 0], c[-1, 1], c[-1, 2], snippet, fontsize=8, color='black')
    start += n

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('3D hidden-state trajectories for prompt sets\nPCA EVR: ' + ', '.join(f'{v:.2%}' for v in explained))
ax.view_init(elev=24, azim=45)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'paper_prompt_trajectories_3d.png', bbox_inches='tight')
plt.show()

# Print prompts and dataset note for documentation.
print('Dataset / benchmark note: SST-2-style sentiment prompts are used for the hidden-state trajectory study; the residual summary is computed from a small SST-2 subset from the Hugging Face GLUE dataset.')
for label, prompt in prompt_specs:
    print(f'{label}: {prompt}')


## Exact prompts used in the hidden-state trajectory plot

1. "I absolutely loved this movie; the acting was brilliant and the story was moving."
2. "This was a terrible experience, and I hated every minute of it."
3. "The service was surprisingly smooth and the staff were helpful."
4. "The product arrived broken and the support was completely useless."

These are grouped into positive and negative prompt sets only for visualization clarity; they are not a distinct model-native concept.

In [ ]:
# Layer-group summary: early / middle / late residual means with error bars
summary_df = pd.DataFrame(summary).sort_values('layer').reset_index(drop=True)
layer_count = len(summary_df)
layer_groups = {
    'Early': summary_df.iloc[: max(1, layer_count // 3)],
    'Middle': summary_df.iloc[max(1, layer_count // 3): min(layer_count - 1, 2 * layer_count // 3)],
    'Late': summary_df.iloc[max(1, 2 * layer_count // 3):],
}

bar_labels, bar_means, bar_stds = [], [], []
for name, group in layer_groups.items():
    if group.empty:
        continue
    bar_labels.append(name)
    bar_means.append(float(group['mean_relative_residual'].mean()))
    bar_stds.append(float(group['std_relative_residual'].mean()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: layer-wise residual curve
ax0 = axes[0]
ax0.plot(
    summary_df['layer'],
    summary_df['mean_relative_residual'],
    color='#1f77b4',
    linewidth=2.5,
    marker='o',
    markersize=5,
    label='mean relative residual'
)
ax0.fill_between(
    summary_df['layer'],
    summary_df['mean_relative_residual'] - summary_df['std_relative_residual'],
    summary_df['mean_relative_residual'] + summary_df['std_relative_residual'],
    color='#1f77b4',
    alpha=0.15,
)
ax0.axvline(summary_df['layer'].iloc[0], linestyle='--', color='gray', alpha=0.4)
ax0.axvline(summary_df['layer'].iloc[-1], linestyle='--', color='gray', alpha=0.4)
ax0.set_xlabel('Layer index')
ax0.set_ylabel('Mean relative residual')
ax0.set_title('Residual dynamics across layers')
ax0.grid(alpha=0.25)
ax0.legend(loc='upper right')

# Right: early / middle / late group means with bars and error bars
ax1 = axes[1]
ax1.bar(
    bar_labels,
    bar_means,
    yerr=bar_stds,
    capsize=6,
    color=['#4c72b0', '#dd8452', '#55a868'],
    alpha=0.85,
    edgecolor='black',
)
ax1.set_ylabel('Average relative residual')
ax1.set_title('Residual by layer regime')
ax1.grid(axis='y', alpha=0.25)
for i, v in enumerate(bar_means):
    ax1.text(i, v + max(bar_stds) * 0.08, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

fig.tight_layout()
fig.savefig(PLOTS_DIR / 'residual_group_summary.png', dpi=200, bbox_inches='tight')
plt.show()

# Actual hidden-state dynamics vs linearized prediction from the local Jacobian
prompt_label, prompt = prompt_specs[0]
encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=32)
encoded = {k: v.to(device) for k, v in encoded.items()}

with torch.no_grad():
    outputs = model(**encoded, output_hidden_states=True, return_dict=True, use_cache=False)

layer_inputs = capture_layer_inputs(model, encoded)
actual_norms = []
predicted_norms = []
layer_ids = []

for layer_idx in range(len(model.model.layers)):
    actual_vec = outputs.hidden_states[layer_idx + 1][0, -1, :].float().cpu()
    input_tensor, layer_kwargs = layer_inputs[layer_idx]
    jacobian = layer_last_token_jacobian(
        model.model.layers[layer_idx],
        input_tensor,
        layer_kwargs,
        vjp_chunk_size=64,
    ).to(device)
    pred_vec = (jacobian @ input_tensor[0, -1, :].to(device)).detach().cpu()
    actual_norms.append(float(torch.linalg.norm(actual_vec)))
    predicted_norms.append(float(torch.linalg.norm(pred_vec)))
    layer_ids.append(layer_idx)

fig2, ax2 = plt.subplots(figsize=(10, 5))
ax2.plot(layer_ids, actual_norms, color='#1f77b4', linewidth=2.5, marker='o', markersize=4, label='actual h_l norm')
ax2.plot(layer_ids, predicted_norms, color='#d62728', linewidth=2.5, linestyle='--', marker='s', markersize=4, label='linearized h_l norm')
ax2.set_xlabel('Layer index')
ax2.set_ylabel('Hidden-state norm ||h_l||')
ax2.set_title(f'Activation vs Jacobian-predicted hidden-state dynamics\nPrompt: {prompt[:60]}')
ax2.grid(alpha=0.25)
ax2.legend(loc='best')
fig2.tight_layout()
fig2.savefig(PLOTS_DIR / 'actual_vs_linearized_hidden_dynamics.png', dpi=200, bbox_inches='tight')
plt.show()

print('Saved group summary figure to:', PLOTS_DIR / 'residual_group_summary.png')
print('Saved actual-vs-linearized dynamics figure to:', PLOTS_DIR / 'actual_vs_linearized_hidden_dynamics.png')


In [ ]:
# Final combined figure: residuals + prompt panel + hidden-state dynamics + 3D trajectories
# Two-panel export: (1) prompt-conditioned view, (2) repeated-same-prompt stability view

from matplotlib.patches import FancyBboxPatch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# ---------- helpers ----------

def gather_single_prompt_dynamics(model, tokenizer, prompt, device, max_length=32):
    encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=max_length)
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        outputs = model(**encoded, output_hidden_states=True, return_dict=True, use_cache=False)
    layer_inputs = capture_layer_inputs(model, encoded)
    actual_traj = []
    linear_traj = []
    rel_res = []
    for layer_idx in range(len(model.model.layers)):
        h_act = outputs.hidden_states[layer_idx + 1][0, -1, :].float().cpu()
        actual_traj.append(h_act.numpy())
        input_tensor, layer_kwargs = layer_inputs[layer_idx]
        jacobian = layer_last_token_jacobian(
            model.model.layers[layer_idx],
            input_tensor,
            layer_kwargs,
            vjp_chunk_size=64,
        ).to(device)
        pred_vec = (jacobian @ input_tensor[0, -1, :].to(device)).detach().cpu()
        linear_traj.append(pred_vec.numpy())
        next_norm = torch.linalg.norm(h_act)
        residual_vec = h_act - pred_vec
        rel_res.append(float(torch.linalg.norm(residual_vec) / max(float(next_norm), 1e-12)))
    return {
        'traj': np.stack(actual_traj, axis=0),
        'pred_traj': np.stack(linear_traj, axis=0),
        'rel_res': np.array(rel_res, dtype=float),
        'layer_idx': np.arange(len(model.model.layers)),
    }

# ---------- main prompt-conditioned panel ----------
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map='auto',
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

prompt_specs = [
    ('Prompt set 1 (positive)', 'I absolutely loved this movie; the acting was brilliant and the story was moving.'),
    ('Prompt set 2 (positive)', 'The performance was outstanding, and the atmosphere felt warm and uplifting.'),
    ('Prompt set 3 (positive)', 'The service was surprisingly smooth and the staff were helpful.'),
    ('Prompt set 4 (positive)', 'This product is excellent, reliable, and genuinely improved my day.'),
    ('Prompt set 5 (positive)', 'The experience was delightful from start to finish and felt thoughtfully designed.'),
    ('Prompt set 6 (negative)', 'This was a terrible experience, and I hated every minute of it.'),
    ('Prompt set 7 (negative)', 'The product arrived broken and the support was completely useless.'),
    ('Prompt set 8 (negative)', 'I felt frustrated and disappointed by the entire process and quality.'),
    ('Prompt set 9 (negative)', 'Everything felt slow, confusing, and badly handled from beginning to end.'),
    ('Prompt set 10 (negative)', 'The service was awful and the final result left me deeply dissatisfied.'),
]

prompt_dynamics = [gather_single_prompt_dynamics(model, tokenizer, prompt, device) for _, prompt in prompt_specs]
all_actual = np.vstack([d['traj'] for d in prompt_dynamics])
all_actual_centered = all_actual - all_actual.mean(axis=0, keepdims=True)
proj = PCA(n_components=3, random_state=0)
coords = proj.fit_transform(all_actual_centered)
expl = proj.explained_variance_ratio_

# residual layer summary from stored df
summary_df = pd.DataFrame(summary).sort_values('layer').reset_index(drop=True)
layer_count = len(summary_df)
layer_groups = {
    'Early': summary_df.iloc[:max(1, layer_count // 3)],
    'Middle': summary_df.iloc[max(1, layer_count // 3):min(layer_count - 1, 2 * layer_count // 3)],
    'Late': summary_df.iloc[max(1, 2 * layer_count // 3):],
}
bar_labels, bar_means, bar_stds = [], [], []
for name, group in layer_groups.items():
    if group.empty:
        continue
    bar_labels.append(name)
    bar_means.append(float(group['mean_relative_residual'].mean()))
    bar_stds.append(float(group['std_relative_residual'].mean()))

# Build final figure with four panels
fig = plt.figure(figsize=(16, 12), constrained_layout=True)
# panel arrangement
ax_res = fig.add_subplot(2, 2, 1)
ax_prompt = fig.add_subplot(2, 2, 2)
ax_dyn = fig.add_subplot(2, 2, 3)
ax_3d = fig.add_subplot(2, 2, 4, projection='3d')

# 1) residual vs layer
ax_res.plot(summary_df['layer'], summary_df['mean_relative_residual'], color='#1f77b4', linewidth=2.5, marker='o', markersize=5)
ax_res.fill_between(
    summary_df['layer'],
    summary_df['mean_relative_residual'] - summary_df['std_relative_residual'],
    summary_df['mean_relative_residual'] + summary_df['std_relative_residual'],
    color='#1f77b4', alpha=0.18,
)
ax_res.set_xlabel('Layer index')
ax_res.set_ylabel('Mean relative residual')
ax_res.set_title('Residual dynamics across layers')
ax_res.grid(alpha=0.25)

# 2) prompt list panel
prompt_text = '\n'.join([f'{idx}. {prompt[:90]}' for idx, (_, prompt) in enumerate(prompt_specs, start=1)])
ax_prompt.axis('off')
textbox = ax_prompt.text(
    0.02, 0.98, 'Prompts used\n\n' + prompt_text,
    va='top', ha='left', fontsize=9, family='monospace', transform=ax_prompt.transAxes,
    bbox=dict(facecolor='white', edgecolor='0.7', boxstyle='round,pad=0.5', alpha=0.95),
)
ax_prompt.set_title('Prompt panel')

# 3) actual vs predicted hidden-state norm curve
all_layer_ids = np.arange(len(model.model.layers))
# representative prompt: first positive prompt for panel
rep = prompt_dynamics[0]
ax_dyn.plot(all_layer_ids, np.linalg.norm(rep['traj'], axis=1), color='#1f77b4', linewidth=2.5, marker='o', label='actual h_l norm')
ax_dyn.plot(all_layer_ids, np.linalg.norm(rep['pred_traj'], axis=1), color='#d62728', linewidth=2.5, linestyle='--', marker='s', label='linearized h_l norm')
ax_dyn.set_xlabel('Layer index')
ax_dyn.set_ylabel('||h_l||')
ax_dyn.set_title('Actual vs. Jacobian-predicted hidden-state norm')
ax_dyn.grid(alpha=0.25)
ax_dyn.legend(loc='best')

# 4) 3D PCA hidden-state trajectories with printed prompts
start = 0
for traj, (label, prompt) in zip([d['traj'] for d in prompt_dynamics], prompt_specs):
    end = start + len(traj)
    c = coords[start:end]
    pos = 'positive' in label.lower()
    color = '#1f77b4' if pos else '#d62728'
    ax_3d.plot(c[:, 0], c[:, 1], c[:, 2], color=color, linewidth=2, marker='o', markersize=4, alpha=0.9)
    ax_3d.scatter([c[0, 0]], [c[0, 1]], [c[0, 2]], color='black', s=18, zorder=5)
    ax_3d.scatter([c[-1, 0]], [c[-1, 1]], [c[-1, 2]], color='black', s=24, zorder=5)
    snippet = (prompt[:36] + '...') if len(prompt) > 36 else prompt
    ax_3d.text(c[-1, 0], c[-1, 1], c[-1, 2], snippet, fontsize=8, color='black')
    start = end
ax_3d.set_xlabel('PC1')
ax_3d.set_ylabel('PC2')
ax_3d.set_zlabel('PC3')
ax_3d.set_title('3D hidden-state trajectories\nPCA EVR: ' + ', '.join(f'{v:.2%}' for v in expl))
ax_3d.view_init(elev=22, azim=40)

fig.savefig(PLOTS_DIR / 'final_prompt_state_and_residual_summary.png', dpi=200, bbox_inches='tight')
plt.show()

# ---------- repeated same prompt stability panel ----------
repeat_prompt = 'I absolutely loved this movie; the acting was brilliant and the story was moving.'
repeat_results = [gather_single_prompt_dynamics(model, tokenizer, repeat_prompt, device) for _ in range(10)]
repeat_actual = np.stack([d['traj'] for d in repeat_results], axis=0)  # [repeat, layer, dim]
repeat_res = np.stack([d['rel_res'] for d in repeat_results], axis=0)  # [repeat, layer]
repeat_x = repeat_actual.reshape(-1, repeat_actual.shape[-1])
repeat_centered = repeat_x - repeat_x.mean(axis=0, keepdims=True)
repeat_proj = PCA(n_components=3, random_state=0)
repeat_coords = repeat_proj.fit_transform(repeat_centered)
repeat_expl = repeat_proj.explained_variance_ratio_
repeat_layer_mean = repeat_res.mean(axis=0)
repeat_layer_std = repeat_res.std(axis=0, ddof=0)

fig2 = plt.figure(figsize=(16, 10), constrained_layout=True)
ax20 = fig2.add_subplot(2, 2, 1)
ax21 = fig2.add_subplot(2, 2, 2, projection='3d')
ax22 = fig2.add_subplot(2, 2, 3)
ax23 = fig2.add_subplot(2, 2, 4)

# panel A: repeated path residuals, with mean and std band
for idx in range(repeat_res.shape[0]):
    ax20.plot(np.arange(repeat_res.shape[1]), repeat_res[idx], color='#7f7f7f', alpha=0.25, linewidth=1.2)
ax20.plot(np.arange(repeat_res.shape[1]), repeat_layer_mean, color='#1f77b4', linewidth=2.5, marker='o', label='mean residual')
ax20.fill_between(np.arange(repeat_res.shape[1]), repeat_layer_mean - repeat_layer_std, repeat_layer_mean + repeat_layer_std, color='#1f77b4', alpha=0.18)
ax20.set_xlabel('Layer index')
ax20.set_ylabel('Relative residual')
ax20.set_title('Same prompt repeated 10x: residual stability')
ax20.grid(alpha=0.25)

# panel B: same prompt repeated 10x in 3D PCA
for r in range(repeat_actual.shape[0]):
    c = repeat_coords[r * repeat_actual.shape[1]:(r + 1) * repeat_actual.shape[1]]
    ax21.plot(c[:, 0], c[:, 1], c[:, 2], color='#1f77b4', alpha=0.55, linewidth=1.7)
ax21.set_xlabel('PC1')
ax21.set_ylabel('PC2')
ax21.set_zlabel('PC3')
ax21.set_title('Repeated prompt trajectories\nPCA EVR: ' + ', '.join(f'{v:.2%}' for v in repeat_expl))
ax21.view_init(elev=22, azim=40)

# panel C: actual vs predicted norm for a representative repeat
rep_repeat = repeat_results[0]
ax22.plot(np.arange(len(model.model.layers)), np.linalg.norm(rep_repeat['traj'], axis=1), color='#1f77b4', marker='o', linewidth=2.5, label='actual h_l norm')
ax22.plot(np.arange(len(model.model.layers)), np.linalg.norm(rep_repeat['pred_traj'], axis=1), color='#d62728', linestyle='--', marker='s', linewidth=2.5, label='linearized h_l norm')
ax22.set_xlabel('Layer index')
ax22.set_ylabel('||h_l||')
ax22.set_title('Repeated prompt: actual vs predicted hidden-state norm')
ax22.grid(alpha=0.25)
ax22.legend(loc='best')

# panel D: bar plot of residual variability for early / middle / late layers
layer_regime = {
    'Early': slice(0, max(1, repeat_res.shape[1] // 3)),
    'Middle': slice(max(1, repeat_res.shape[1] // 3), min(repeat_res.shape[1] - 1, 2 * repeat_res.shape[1] // 3)),
    'Late': slice(max(1, 2 * repeat_res.shape[1] // 3), repeat_res.shape[1]),
}
reg_means, reg_stds = [], []
for name, sl in layer_regime.items():
    vals = repeat_res[:, sl]
    reg_means.append(float(vals.std(axis=0).mean()))
    reg_stds.append(float(vals.std(axis=0).std()))
ax23.bar(['Early', 'Middle', 'Late'], reg_means, yerr=reg_stds, capsize=6, color=['#4c72b0', '#dd8452', '#55a868'], edgecolor='black')
ax23.set_ylabel('Residual variability across repeats')
ax23.set_title('Single-prompt repeat stability (std across 10 repeats)')
ax23.grid(axis='y', alpha=0.25)
for i, v in enumerate(reg_means):
    ax23.text(i, v + max(reg_stds) * 0.08, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

fig2.savefig(PLOTS_DIR / 'repeated_prompt_stability_summary.png', dpi=200, bbox_inches='tight')
plt.show()

print('Saved combined prompt figure:', PLOTS_DIR / 'final_prompt_state_and_residual_summary.png')
print('Saved repeated-prompt stability figure:', PLOTS_DIR / 'repeated_prompt_stability_summary.png')
print('3D prompt EVR:', expl)
print('Repeated prompt EVR:', repeat_expl)


In [ ]:
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')

within_curve_matrix = np.stack([residual_curve_for_prompt(prompt) for prompt in within_prompts], axis=0)
ood_curve_matrix = np.concatenate([
    np.stack([residual_curve_for_prompt(prompt) for prompt in prompts], axis=0)
    for prompts in ood_prompt_bank.values()
], axis=0)

layer_count = within_curve_matrix.shape[1]
regimes = ['early', 'mid', 'late']
colors = ['#2A9D8F', '#F4A261']

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
fig.patch.set_facecolor('#f8f8f8')

for ax, regime in zip(axes, regimes):
    sl = regime_slice(layer_count, regime)
    within_vals = within_curve_matrix[:, sl].mean(axis=1)
    ood_vals = ood_curve_matrix[:, sl].mean(axis=1)

    positions = [0, 1]
    dataset = [within_vals, ood_vals]
    violin = ax.violinplot(
        dataset,
        positions=positions,
        widths=0.42,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )

    for body, color in zip(violin['bodies'], colors):
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.82)
        body.set_linewidth(1.4)

    for pos, vals, color in zip(positions, dataset, colors):
        jitter = np.linspace(-0.06, 0.06, len(vals))
        ax.scatter(
            np.full(len(vals), pos) + jitter,
            vals,
            s=28,
            color=color,
            edgecolors='black',
            linewidths=0.5,
            alpha=0.9,
            zorder=3,
        )
        mean_val = float(np.mean(vals))
        ax.scatter(
            pos,
            mean_val,
            s=72,
            marker='o',
            facecolor='white',
            edgecolor='black',
            linewidth=1.3,
            zorder=4,
        )

    _, p_value = stats.ttest_ind(within_vals, ood_vals, equal_var=False, nan_policy='omit')
    sig_text = '★' if p_value < 0.05 else 'n.s.'
    y_top = max(float(np.max(within_vals)), float(np.max(ood_vals))) * 1.12
    ax.text(0.5, y_top, sig_text, ha='center', va='bottom', fontsize=14, fontweight='bold')

    ax.set_title(f'{regime.title()} layers', fontsize=13, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Within', 'OOD'])
    ax.set_ylabel('Residual magnitude', fontsize=11)
    ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)
    ax.set_axisbelow(True)
    ax.set_ylim(0, max(0.05, max(float(np.max(within_vals)), float(np.max(ood_vals))) * 1.6))

fig.suptitle('Within-distribution vs OOD residuals by layer regime', fontsize=16, fontweight='bold', y=1.03)
fig.savefig(PLOTS_DIR / 'within_vs_ood_by_layer_regime.png', dpi=220, bbox_inches='tight')
plt.show()

print('Saved within-vs-OOD comparison:', PLOTS_DIR / 'within_vs_ood_by_layer_regime.png')
print('OOD families used:', ', '.join(ood_prompt_bank.keys()))